# Functions

One of the greatest ability of the human mind is the ability to abstract - and what could be a better concept of abstraction then the concept of functions? Functions help us compact infinite information into a finite space. 

For example, consider the very loved function, $F(x) = x^2$. If we consider only positive integer values for x, the information contained in this function is equivalent to the following table:

| Value of x | Value of F(x) |
| ---------- | ------------- |
| 1 | 1 |
| 2 | 4 |
| 3 | 9 |
| 4 | 16 | 
| ... | ... |

... we can keep filling this table all the way to $+\infty$. This function, **F(x)**, essentially converts this infinite length table in a simple one line statement: $F(x) = x^2$.

This is is the power of functons. Functions are able to abstract out a very large (or even $+\infty$) **state spaces** to a finite and consise form. 

This gives us the motivation to develop function-based Learning Methods where we replace the **Q Table** (which is a table with state-action pairs) to a **Q Function** - which is a function that aims to compact the Table of Q into a simple and concise statement form.

## Baby Steps

First lets consider our favourite, Monte Carlo Update Algorithm!

The Update equation is:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big( G_t - Q(s, a) \Big)$$


Where Q(s,a) is a Q Table which may like:

| State s | Action a | Value of Q(s, a) |
| ---------- | ------------- | --- |
| 1 | 1 | 0.4 | 
| 1 | 2 | - 0.2 | 
| 1 | 3 | 4 |
| 1 | 4 | 9 |
| 2 | 1 | 67 |
| 2 | 2 | 2 |
| 2 | 3 | 4 |
| 2 | 4 |  5 |
| ... | ... | ...|

Obviously the values of Q(s,a) is arbitrary in this table. But the size of this table will be | S | * | A | where S is the set of states and A is the Set of Actions. For cliff walking, this table will have a size of ( 37 ) * ( 4 ) = 148 distinct rows.

Imgine, instead of having 37 states, we had 10,000 states and 200 actions. We would have 2,000,000 distinct rows whose Q(s, a) we will have to "learn" by interacting with the environment. Firstly, that would take a long time to update accurate values, and, in some cases might even be impossible to learn in one human lifetime (80 years). Another issue is the issue of constantly keeping this much information in memory - 2,000,000 distict rows is no joke to store.

This is where we can use functions to *approximate* the values of Q(s, a) instead of storing the entire table, we will only store the function and calculate the actual Q-Values in runtime!

So, consider the following Q Function:

$$Q_{function} (w, a, s) = F(w, a, s) $$

Where w is a vector (which is a learnable parameter), and assume F(w, a, s) gives us the correct value of taking action, a, at state s for any valid values of a and s.

The our policy can be rewritten from:

$$
\pi_{*} = \argmax_{a} (Q_{table}(a, s));


\forall a \in A, s \in S
$$

to

$$
\pi_{*} = \argmax_{a} (Q_{function}(w, a, s));
\forall a \in A, s \in S
$$ 

when Q-Table and $Q_{function} are optimal.

However, our problem then becomes how can I update the value of w s.t F(w, a, s) will always output the correct Q(s, a) Value. Moreover, how can we decide how F(w, a, s) will look like?



### How does F(w, a, s) look like?

The straightforward answer is - *we don't know*. There is probably no universal function, F(w, a, s), that will work for solving all problems. But we can take guess a function that can work and check if it is able to correctly estimate values of Q(s, a). Lets look at some candidate functions for our table:

#### Linear Function

$F(w, a, s) = w_{1} * s + w_{2} * s + w_{3} * s + w_{4} * s + a$

Where $w_k$ is the $k-th$ element of vector $w$

For a 4 dimentional vector w. Of course, we can use more dimentions for vector w and see if it can work. 

---

#### Power Function
$ F(w, a, s) = w_{1} * s + w_{2} * s^2 + w_{3} * s^3 + w_{4} * s^4 + a$

For a 4 dimentional vector w. Of course, we can use more dimentions for vector w and see if it can work.

---


#### Core Properties

The core properties for the candidate function F(w, a, s) is that it can:

1. Domain of s, is greater than or requal to the set, S, which contains all values of s. (Trivial)
2. Domain of a, is greater than or requal to the set, A, which contains all values of a. (Trivial)
3. Correctly able to map state-action space to value space (Difficult)

The third point is the one that is diffult to satisfy. We have to perform trial and error to really know which function works.


## Optimise F

Once we have a candidate Fuction, we need to optimise w. We want to reduce the error:

$$ min_{w}(G_t - F(w, s, a)) $$

Where $G_t$ is the actual return we get at state s, taking action a and following the our policy $\pi$ until termination. However, instead of minsing $ (G_t - F(w, s, a)) $, we should mininmise:

$$ min_{w} [G_t - F(w, s, a)]^2 $$

Because it will penalise greater deviations more. Furthermore, we are not trying to minimise our Error at just state s and a, but for all states and all actions. We can write all that down succintly by saying:

$$ min_{w} \sum_{a , s} [G_t - F(w, s, a)]^2 ; \forall a \in A, s \in S  $$


### Minimise

Now let's actually perform minization. Easy way to do this, is simply differentiating F and then following **down-slope**, of course, given F is differentiable. Which would mean:

$$ w \leftarrow w - \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$

Note that if we want to go above slope, we would use a **plus** (+) sign instead of **minus** (-)

Now lets incorporate a learning rate, call it $\alpha$ and introduce a constant scaling of $\frac{1}{2}$. Our equation will become:

$$w \leftarrow w - \alpha \frac{1}{2} \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$

Lets simplify by performing the differntiation:

$$   \alpha \frac{1}{2} \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$
$$ =  \alpha \frac{1}{2} \sum_{a , s} 2 * [G_t - F(w, s, a)] * (-1) \frac{\partial}{\partial w} F(w, s, a) $$
$$ = \alpha \frac{1}{2} * 2 * (-1) \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$
$$ =  - \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$


Substituting the above differentiation to our optimization equation, it becomes:

$$ w \leftarrow w - ( - \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)] )$$

or, simply

$$ w \leftarrow w + \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$



### Limitations

Notice that we now have limitations with $ F(w, s, a)$ since we have to calculate $\frac{\partial}{\partial w} F(w, s, a)$ which means our q-function, F, has to be differentiable. Another limitation is that we need the complete return, $G_t$, of an episode before we start making updates to our q-function. So our Agent will be acting completely randomly on the first episode and if the episodes are very long, our agent will take a long time to converge to optimal policy. 

Keeping these limitations in mind, let's see an example on how we can implement a functional approach to RL instead of the traditional q tables.

### Implementation

Instead of worrying about whether our function will be able to effectively model the environment, lets use a fool proof function to do it! Note that for cliff walking there are less than 48 states (4*12 = 48).

Define:
$$ F(w,s,a) = w_a * x(s) $$

s.t 

$$ \frac{\partial}{\partial w} F(w,s,a) =  x(s) $$

Where:
- $w_a$ is a 48-dimensional vector depending on value of a. so if we have $w_0$, $w_1$, $w_2$ and $w_3$
- x(s) is a 48-dimensional vector that depends on s. We define x(s) as:

$$ x_k = \begin{cases} \text{1} &  \text{if k = s } \\ 
\text{0} &  \text{if k } {\neq s } \end{cases} 
\quad \text{; where } x_k \text{ is the } k^{th} \text{term in } x(s) 
$$

in code it may look like:

```python

def x(s):
    temp = np.zeros(48) # 48 columns
    temp[s] = 1
    return temp

w = np.zeros((4, 48)) # 48 cols, 4 rows

def F(w, s, a):
    weights = w[a]
    result = np.dot(weights, x(s))

    return result


def dwF(w, s, a):
    return x(s)
```

This is a very popular candidate Q-Function and is known as Linear Value Function Approximation with one-hot state encoding.

### Implementation

Lets actually implement this in python!

In [38]:
from utils import *
import numpy as np
import gymnasium as gym

epsilon = 0.1
epsilon_decay_rate = 0.999
epsilon_min = 0.01

gemma = 0.9
alpha = 0.1
num_episodes = 5000
MAX_STEPS_PER_EPISODE = 500

seed = 42

env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)

weights = np.zeros((len(action_space), len(observation_space)))

def x(state):
    temp = np.zeros(len(observation_space))
    temp[state] = 1
    return temp

def q_function(weights, state, action):
    weight_vector = weights[action]
    value = np.dot(weight_vector, x(state=state))
    return value

def q_function_differential(weights, state, action):
    return x(state=state)


episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []

In [39]:
rng = np.random.default_rng(seed=seed)


def get_best_acton(weights, state):
    num_actions = weights.shape[0]
    best_action_idx = 0
    best_action_val = weights[0, state]
    for action in range(num_actions):
        if best_action_val < weights[action, state]:
            best_action_idx = action
            best_action_val = weights[action, state]

    return best_action_idx


for episode in range(num_episodes):

    start_time = datetime.now()
    env.reset()

    # Array to store, State, Action, Reward for the current episode. WE use it to backtrack and update the Q-values at the end of the episode.
    episode_data = []
    terminated = False
    truncated = False

    state, info = env.reset()
    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)  # Decay epsilon after each episode

    while not (terminated or truncated):

        # Epsilon-greedy action selection
        if rng.random() < epsilon:
            action = rng.choice(action_space)
        else:
            action = get_best_acton(weights=weights, state=state)

        next_state, reward, terminated, truncated, info = env.step(action)

        episode_data.append((state, action, reward))

        state = next_state

    # Calculate returns and update Q-Function weights
    G = 0
    for state, action, reward in reversed(episode_data):
        G = gemma * G + reward
        TD_Error = G - q_function(weights=weights, state=state, action=action)
        weights[action] += alpha * TD_Error * q_function_differential(weights=weights, state=state, action=action)


    # Average episode error for monitoring convergence
    Avg_MC_Episode_Error = np.mean([np.abs(G - q_function(weights=weights, state=state, action=action)) for state, action, reward in episode_data])
    episode_avg_mc_error.append(Avg_MC_Episode_Error)


    end_time = datetime.now()
    episode_duration = (end_time - start_time).total_seconds()
    episode_durations.append(episode_duration)
    episode_rewards.append(sum([reward for _, _, reward in episode_data]))
    episode_lengths.append(len(episode_data))


    if (episode + 1) % 200 == 0:
        print(f"Episode: {episode + 1}, Average MC Episode Error: {Avg_MC_Episode_Error:.4f}",
                f"Episode Duration: {episode_duration:.4f} seconds",
                f"Total Reward: {episode_rewards[-1]}, Episode Length: {episode_lengths[-1]}",
                f"Epsilon: {epsilon:.4f}")

Episode: 200, Average MC Episode Error: 2.4548 Episode Duration: 0.0003 seconds Total Reward: -26, Episode Length: 26 Epsilon: 0.0819
Episode: 400, Average MC Episode Error: 2.5603 Episode Duration: 0.0003 seconds Total Reward: -21, Episode Length: 21 Epsilon: 0.0670
Episode: 600, Average MC Episode Error: 2.6265 Episode Duration: 0.0007 seconds Total Reward: -23, Episode Length: 23 Epsilon: 0.0549
Episode: 800, Average MC Episode Error: 2.5520 Episode Duration: 0.0002 seconds Total Reward: -21, Episode Length: 21 Epsilon: 0.0449
Episode: 1000, Average MC Episode Error: 2.6039 Episode Duration: 0.0002 seconds Total Reward: -21, Episode Length: 21 Epsilon: 0.0368
Episode: 1200, Average MC Episode Error: 0.0236 Episode Duration: 0.0051 seconds Total Reward: -500, Episode Length: 500 Epsilon: 0.0301
Episode: 1400, Average MC Episode Error: 0.0135 Episode Duration: 0.0051 seconds Total Reward: -500, Episode Length: 500 Epsilon: 0.0246
Episode: 1600, Average MC Episode Error: 0.9170 Episode

In [ ]:
import torch
from torch import nn


def func(x):
    return x*2

input_dim = 1
output_dim = 1

class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.fc = nn.Linear(input_dim, 3)
        
